# Headroom re-run — Qwen3-4B

Repeat of the headroom check on the new model, per `DECISION_model_change.md`.

**Threshold is unchanged: 70% on `strict`.** Amendment 001's rules are unchanged. Neither was touched when the model changed — see the decision record §6.

**Before running:** commit the signed `DECISION_model_change.md`. Its argument has to stand without reference to any number this notebook produces, and committing it first is what shows that.

### What's different from last time

Last time the headroom check ran via `!python` and the diagnostic ran in the notebook kernel — two model copies, one T4, out of memory. This notebook does both in **one script, one process**. No OOM, and one model load instead of two.

Outputs go to `results/raw/headroom_qwen3-4b/` so the Qwen3.5-4B results are not overwritten. Those are kept and reported alongside.

---
### Turn the GPU on

`Runtime` → `Change runtime type` → **T4 GPU** → Save.

## Cell 1 — GPU check

In [ ]:
import torch

print("GPU visible:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))
    free, total = torch.cuda.mem_get_info()
    print(f"Free: {free/1e9:.1f} GB of {total/1e9:.1f} GB")
    if free / total < 0.9:
        print("\n>>> Memory already in use. Runtime > Restart session before continuing.")
else:
    print("\n>>> STOP. Runtime > Change runtime type > T4 GPU > Save, then re-run.")

## Cell 2 — Install and fetch the prompts

In [ ]:
!pip -q install -U transformers accelerate
!git clone -q --depth 1 https://github.com/anthropics/jacobian-lens.git

import json
items = json.load(open("jacobian-lens/data/experiments/probe-swap.json"))["items"]
print(f"{len(items)} prompts loaded")

## Cell 3 — Write the script

Headroom check and diagnostic combined. Same scoring as the Qwen3.5-4B run: prompt right-stripped, answer given a leading space, `strict` = greedy continuation truncated to the answer's token length.

In [ ]:
%%writefile headroom_full.py
"""Headroom check + diagnostic in ONE process.

Combines what were previously two runs (headroom_check.py via !python, and an
in-notebook diagnostic cell). Running both in one session loaded the model
twice and exhausted a T4. One script, one model load, three output files.

Scoring is UNCHANGED from the Qwen3.5-4B run:
  - prompt.rstrip(), answer = " " + answer.strip()   (the trailing-space fix)
  - strict := greedy continuation truncated to the answer's token length
Amendment 001's rules are NOT applied here. rescore.py stays a separate,
frozen, auditable step operating on diagnostic_rows.json.

Usage:
    python headroom_full.py --model Qwen/Qwen3-4B --data <probe-swap.json> \
        --out results/raw/headroom_qwen3-4b/ --dtype float16
"""
from __future__ import annotations
import argparse, json, subprocess
from collections import Counter, defaultdict
from pathlib import Path

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

EXTRA_TOKENS = 6


def wilson(k: int, n: int, z: float = 1.96) -> tuple[float, float]:
    if n == 0:
        return (0.0, 0.0)
    p = k / n
    d = 1 + z**2 / n
    c = (p + z**2 / (2 * n)) / d
    h = z * ((p * (1 - p) / n + z**2 / (4 * n**2)) ** 0.5) / d
    return (max(0.0, c - h), min(1.0, c + h))


@torch.no_grad()
def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", required=True)
    ap.add_argument("--data", required=True)
    ap.add_argument("--out", required=True)
    ap.add_argument("--dtype", default="float16")
    ap.add_argument("--limit", type=int, default=None)
    args = ap.parse_args()

    device = "cuda" if torch.cuda.is_available() else "cpu"
    tok = AutoTokenizer.from_pretrained(args.model)
    model = AutoModelForCausalLM.from_pretrained(
        args.model, dtype=getattr(torch, args.dtype), device_map=device
    ).eval()

    items = json.load(open(args.data))["items"]
    if args.limit:
        items = items[: args.limit]

    rows = []
    for i, it in enumerate(items):
        prompt = it["prompt"].rstrip()
        answer = " " + it["answer"].strip()
        ans_ids = tok(answer, add_special_tokens=False).input_ids
        ids = tok(prompt, return_tensors="pt").input_ids.to(device)

        out = model.generate(
            ids, max_new_tokens=len(ans_ids) + EXTRA_TOKENS,
            do_sample=False, num_beams=1, pad_token_id=tok.eos_token_id,
        )
        gen_ids = out[0, ids.shape[1]:].tolist()
        gen_full = tok.decode(gen_ids)
        gen_trunc = tok.decode(gen_ids[: len(ans_ids)])

        a = answer.strip().lower()
        g = gen_full.strip().lower()
        r = {
            "name": it["name"], "category": it["category"], "answer": it["answer"],
            "n_answer_tokens": len(ans_ids),
            "generated": gen_trunc, "generated_full": gen_full,
            "strict": gen_trunc.strip().lower() == a,
            "exact": gen_trunc.strip().lower() == a,   # alias: same criterion
            "first_token": bool(gen_ids) and gen_ids[0] == ans_ids[0],
            "prefix": g.startswith(a),
            "window": a in g,
        }
        rows.append(r)
        print(f"[{i+1:>2}/{len(items)}] {it['name']:<26} strict={r['strict']:d} "
              f"first={r['first_token']:d}  want={it['answer']!r} got={gen_full!r}")

    n = len(rows)
    k_strict = sum(r["strict"] for r in rows)
    k_first = sum(r["first_token"] for r in rows)
    by_cat = defaultdict(list)
    for r in rows:
        by_cat[r["category"]].append(r["strict"])

    summary = {
        "model": args.model, "n": n,
        "exact": {"k": k_strict, "acc": k_strict / n, "wilson95": wilson(k_strict, n)},
        "first_token": {"k": k_first, "acc": k_first / n, "wilson95": wilson(k_first, n)},
        "prefix": {"k": sum(r["prefix"] for r in rows)},
        "window": {"k": sum(r["window"] for r in rows)},
        "answer_token_lengths": dict(Counter(r["n_answer_tokens"] for r in rows)),
        "by_category_n_ge_4": {
            c: {"n": len(v), "acc": sum(v) / len(v)}
            for c, v in sorted(by_cat.items()) if len(v) >= 4
        },
        "git_commit": subprocess.run(["git", "rev-parse", "HEAD"],
                                     capture_output=True, text=True).stdout.strip() or "UNKNOWN",
        "config": vars(args),
    }

    out_dir = Path(args.out)
    out_dir.mkdir(parents=True, exist_ok=True)
    (out_dir / "rows.json").write_text(json.dumps(rows, indent=2))
    (out_dir / "summary.json").write_text(json.dumps(summary, indent=2))
    (out_dir / "diagnostic_rows.json").write_text(json.dumps(rows, indent=2))

    lo, hi = summary["exact"]["wilson95"]
    print("\n" + "=" * 66)
    print(f"strict / exact : {k_strict}/{n} = {k_strict/n:.1%}   95% CI {lo:.1%}-{hi:.1%}")
    print(f"first_token    : {k_first}/{n} = {k_first/n:.1%}")
    print("=" * 66)
    print("Compare `strict` against the 70% threshold committed 2026-07-27.")
    print("Then run rescore.py on diagnostic_rows.json for the amended number.")
    print("\nby category (n>=4):")
    for c, v in summary["by_category_n_ge_4"].items():
        print(f"   {c:<20} {v['acc']:>6.0%}  (n={v['n']})")


if __name__ == "__main__":
    main()

## Cell 4 — Smoke test, 5 prompts

Downloads the model (~8 GB, a few minutes). Check the generations look like real attempted answers before running all 90.

In [ ]:
!python headroom_full.py \
    --model Qwen/Qwen3-4B \
    --data jacobian-lens/data/experiments/probe-swap.json \
    --out results/smoke_qwen3-4b/ \
    --dtype float16 \
    --limit 5

## Cell 5 — Full run, all 90

In [ ]:
!python headroom_full.py \
    --model Qwen/Qwen3-4B \
    --data jacobian-lens/data/experiments/probe-swap.json \
    --out results/raw/headroom_qwen3-4b/ \
    --dtype float16

## Cell 6 — Amended score

Runs the frozen Amendment 001 rules. `rescore.py` is unchanged from the Qwen3.5-4B run — same three rules, same negator guard, no synonym list.

In [ ]:
from google.colab import files
print("Upload rescore.py from your repo:")
files.upload()

!python rescore.py results/raw/headroom_qwen3-4b/diagnostic_rows.json

## Cell 7 — Download everything

Do this before closing the tab.

In [ ]:
from google.colab import files

for f in ["summary.json", "rows.json", "diagnostic_rows.json"]:
    files.download(f"results/raw/headroom_qwen3-4b/{f}")
files.download("rescored_rows.json")

---
## Then

1. Commit all four JSONs under `results/raw/headroom_qwen3-4b/`.
2. Compare `strict` against 70%.
3. Sign the model-selection decision.
4. Log the session.

**A lower score than Qwen3.5-4B's 63.3% does not reopen the switch.** `DECISION_model_change.md` §6 anticipates that outcome in advance, precisely so it cannot be used as a reason to go back. The response to a low number is a documented partial Control A, or paid compute for an 8B — not a return to the hybrid architecture.